# Proyecto: Campeonato Mundial de F1 - 01. Construcción de la base de datos

### Maestria en Inteligencia Artificial Aplicada UCSM - UTPL

### Curso: Herramientas para Inteligencia Artificial

### Github:
https://github.com/vchirinosb/f1-championship

### Plataformas utilizadas:
- Jupyter Lab
- Kaggle (origen del dataset)
- SQLite

### Librerias utilizadas
- pandas==3.1.1
- sqlalchemy==2.1.9

Este notebook es el primero del proyecto. Revisa el dataset, decide qué archivos se guardan en una base de datos y
cuáles se usan como CSV, construye la base de datos SQLite y comprueba que la carga sea correcta.

Los demás notebooks leen la base de datos que se genera aquí, por eso este debe ejecutarse primero.

## Preparación del entorno

El código compartido del proyecto está en la carpeta `src/f1` y el script de construcción de la base de datos en
`scripts/`. Agregamos ambas carpetas al path de Python para poder importarlos desde el notebook.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

In [2]:
import pandas as pd

import build_database
from f1 import data
from f1.config import CSV_TABLES, DB_TABLES, RAW_DATA_DIR

## Dataset inicial

Usamos el dataset [Formula 1 World Championship (1950 -
2024)](https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020) de Kaggle, **versión 24**.
Contiene los resultados de todas las carreras del campeonato desde 1950 en 14 archivos CSV.

Los 14 CSV originales están incluidos en el repositorio, en la carpeta `data/raw/`, y son el **dataset inicial** del
proyecto. Trabajamos directamente con esos archivos, sin volver a descargarlos: el autor del dataset lo actualiza cada
temporada y así cualquiera que ejecute el notebook obtiene exactamente los mismos datos.

## Archivos del dataset

Leemos cada CSV con Pandas para conocer su tamaño. En los archivos originales los valores nulos aparecen como el texto
`\N`; la función `data.read_raw_csv` los convierte en nulos y pasa los nombres de columna a snake_case (`raceId` →
`race_id`).

In [3]:
files = pd.DataFrame(
    [
        {
            "file": f"{table}.csv",
            "rows": len(frame := data.read_raw_csv(table)),
            "columns": frame.shape[1],
            "nulls": int(frame.isna().sum().sum()),
            "size_kb": round(
                (RAW_DATA_DIR / f"{table}.csv").stat().st_size / 1024
            ),
        }
        for table in DB_TABLES + CSV_TABLES
    ]
).sort_values("rows", ascending=False, ignore_index=True)
files

,file,rows,columns,nulls,size_kb
0,lap_times.csv,589081,6,0,17209
1,driver_standings.csv,34863,7,0,863
2,results.csv,26759,18,122887,1682
3,constructor_standings.csv,13391,7,0,310
4,constructor_results.csv,12625,5,12608,214
5,pit_stops.csv,11371,7,0,433
6,qualifying.csv,10494,9,11668,454
7,races.csv,1125,18,11349,160
8,drivers.csv,861,9,1559,92
9,sprint_results.csv,360,16,73,24


In [4]:
files[["rows", "size_kb"]].sum()

rows       701433
size_kb     21474
dtype: int64

Descripción de cada archivo:

* **circuits**: circuitos donde se ha corrido (nombre, ciudad, país, coordenadas).
* **constructors**: escuderías (McLaren, Ferrari, Red Bull, ...).
* **drivers**: pilotos (nombre, código de tres letras, fecha de nacimiento, nacionalidad).
* **seasons**: temporadas desde 1950.
* **status**: estado de llegada de un piloto (`Finished`, `+1 Lap`, `Engine`, `Accident`, ...).
* **races**: cada Gran Premio (año, fecha dentro de la temporada, circuito, fecha y horarios).
* **results**: resultado de cada piloto en cada carrera (posición de salida, posición final, puntos, vueltas, estado).
  Es la tabla central del dataset.
* **sprint_results**: resultados de las carreras sprint (desde 2021).
* **qualifying**: clasificación del sábado con los tiempos de Q1, Q2 y Q3 (desde 1994).
* **driver_standings** / **constructor_standings**: posición en el campeonato de pilotos y de constructores después de
  cada carrera.
* **constructor_results**: puntos de cada escudería en cada carrera.
* **lap_times**: tiempo de cada vuelta de cada piloto (desde 1996).
* **pit_stops**: cada parada en boxes, con su vuelta y duración (desde 2011).

## Qué tablas van a la base de datos y cuáles quedan como CSV

La actividad pide usar dos fuentes: un dataset en CSV y otro en una base de datos. Repartimos los archivos según su
función:

| Criterio | Tablas en la base de datos | Tablas como CSV |
|---|---|---|
| Archivos | 12: circuits, constructors, drivers, seasons, status, races, results, sprint_results, qualifying, driver_standings, constructor_standings, constructor_results | 2: lap_times, pit_stops |
| Naturaleza | Catálogos y resultados por carrera, relacionados entre sí por claves | Detalle fino: una fila por vuelta o por parada |
| Uso principal | Consultas que unen tablas, por ejemplo *¿quién ganó en Monza en 2019?* une races, circuits, results y drivers | Agregaciones y gráficos de ritmo y estrategia |
| Integridad | Se benefician de claves primarias y foráneas | Solo se leen y se agregan |
| Cobertura | 1950 - 2024 | Desde 1996 (vueltas) y 2011 (paradas) |

`lap_times` tiene por sí solo más filas que todos los demás archivos juntos, pero solo se usa para agregarlo por carrera
y piloto, así que no gana nada al estar en la base de datos.

Como motor elegimos **SQLite**: no necesita instalar un servidor, la base de datos es un único archivo que se sube a
GitHub junto con el SQL, y es suficiente para el volumen del proyecto. Pandas la lee mediante SQLAlchemy con una
conexión de **solo lectura**, así que ni los notebooks, ni la API, ni el asistente LLM pueden modificarla.

## Construcción de la base de datos

El script `scripts/build_database.py`:

1. Crea las tablas con `sql/schema.sql`: claves primarias, claves foráneas e índices. Las columnas quedan en snake_case;
   la columna `rank` de results se llama `fastest_lap_rank` para que su significado sea claro.
2. Carga los 12 CSV en orden: primero las tablas que no dependen de otras (circuitos, pilotos, ...) y después las de
   resultados. No hace falta convertir tipos: SQLite guarda cada valor según el tipo de su columna.
3. Crea las vistas de `sql/views.sql`.
4. Mantiene activas las claves foráneas durante la carga: si una fila apunta a un registro inexistente (por ejemplo, un
   resultado de un piloto que no está en drivers), la carga se detiene.
5. Solo si todo es correcto reemplaza `data/f1.db`. Si algo falla, la base de datos anterior no se modifica.

El script se puede ejecutar varias veces: cada vez reconstruye la base de datos desde cero.

In [5]:
build_database.print_summary(build_database.build_database())

Base de datos creada: data/f1.db (5.9 MB)

Tabla                       Filas
circuits                       77
constructors                  212
drivers                       861
seasons                        75
status                        139
races                       1,125
results                    26,759
sprint_results                360
qualifying                 10,494
driver_standings           34,863
constructor_standings      13,391
constructor_results        12,625

12 tablas cargadas; claves foráneas verificadas.


Tablas y vistas creadas, con su número de columnas:

In [6]:
objects = data.read_sql(
    "SELECT type, name FROM sqlite_master "
    "WHERE type IN ('table', 'view') ORDER BY type, name"
)
objects["columns"] = [
    len(data.read_sql(f"PRAGMA table_info({name})"))
    for name in objects["name"]
]
objects

,type,name,columns
0,table,circuits,9
1,table,constructor_results,5
2,table,constructor_standings,7
3,table,constructors,5
4,table,driver_standings,7
5,table,drivers,9
6,table,qualifying,9
7,table,races,18
8,table,results,18
9,table,seasons,2


Las dos vistas simplifican las consultas más comunes:

* **v_race_results**: el resultado de cada piloto en cada carrera con nombres legibles (carrera, circuito, piloto,
  escudería, estado) en lugar de identificadores.
* **v_race_winners**: el ganador de cada carrera.

## Validación

Comprobamos con Pandas, independientemente del script, que la base de datos tenga los mismos datos que los CSV.

**Filas en la base de datos vs. filas en el CSV**

In [7]:
validation = pd.DataFrame(
    [
        {
            "table": table,
            "csv_rows": len(data.read_raw_csv(table)),
            "db_rows": data.read_sql(
                f"SELECT COUNT(*) AS n FROM {table}"
            )["n"].iloc[0],
        }
        for table in DB_TABLES
    ]
)
validation["match"] = validation["csv_rows"] == validation["db_rows"]
validation

,table,csv_rows,db_rows,match
0,circuits,77,77,True
1,constructors,212,212,True
2,drivers,861,861,True
3,seasons,75,75,True
4,status,139,139,True
5,races,1125,1125,True
6,results,26759,26759,True
7,sprint_results,360,360,True
8,qualifying,10494,10494,True
9,driver_standings,34863,34863,True


**Claves foráneas**: `PRAGMA foreign_key_check` lista las filas que apuntan a un registro inexistente (por ejemplo, un
resultado de un piloto que no está en drivers).

In [8]:
foreign_key_errors = data.read_sql("PRAGMA foreign_key_check")
len(foreign_key_errors)

0

**lap_times y pit_stops no están en la base de datos**: se leen directamente desde el CSV.

In [9]:
tables_in_db = set(objects.loc[objects["type"] == "table", "name"])
{table: table in tables_in_db for table in CSV_TABLES}

{'lap_times': False, 'pit_stops': False}

In [10]:
data.read_lap_times().head()

,race_id,driver_id,lap,position,time,milliseconds
0,841,20,1,1,1:38.109,98109
1,841,20,2,1,1:33.006,93006
2,841,20,3,1,1:32.713,92713
3,841,20,4,1,1:32.803,92803
4,841,20,5,1,1:32.342,92342


## Consultas de ejemplo

Las consultas se ejecutan con `data.read_sql`, que usa una conexión de **solo lectura** y recibe los valores como
parámetros (`:circuit`, `:year`) en lugar de concatenarlos en el texto SQL. El archivo `sql/queries.sql` tiene estas y
otras consultas.

**¿Quién ganó en Monza en 2019?**

In [11]:
data.read_sql(
    "SELECT year, race_name, driver_name, constructor_name FROM v_race_winners "
    "WHERE circuit_ref = :circuit AND year = :year",
    {"circuit": "monza", "year": 2019},
)

,year,race_name,driver_name,constructor_name
0,2019,Italian Grand Prix,Charles Leclerc,Ferrari


**Campeones de pilotos de las últimas temporadas**: el líder del campeonato después de la última carrera del año.

In [12]:
champions = data.read_sql(
    """
    SELECT ra.year, d.forename || ' ' || d.surname AS champion,
           ds.points, ds.wins
    FROM driver_standings AS ds
    JOIN races AS ra ON ra.race_id = ds.race_id
    JOIN drivers AS d ON d.driver_id = ds.driver_id
    WHERE ds.position = 1
      AND ra.round = (SELECT MAX(round) FROM races WHERE year = ra.year)
    ORDER BY ra.year DESC
    """
)
champions.head(10)

,year,champion,points,wins
0,2024,Max Verstappen,437.0,9
1,2023,Max Verstappen,575.0,19
2,2022,Max Verstappen,454.0,15
3,2021,Max Verstappen,395.5,10
4,2020,Lewis Hamilton,347.0,11
5,2019,Lewis Hamilton,413.0,11
6,2018,Lewis Hamilton,408.0,11
7,2017,Lewis Hamilton,363.0,9
8,2016,Nico Rosberg,385.0,9
9,2015,Lewis Hamilton,381.0,10


**Pilotos con más victorias**

In [13]:
data.read_sql(
    """
    SELECT driver_name, COUNT(*) AS wins
    FROM v_race_winners
    GROUP BY driver_id
    ORDER BY wins DESC
    LIMIT 10
    """
)

,driver_name,wins
0,Lewis Hamilton,105
1,Michael Schumacher,91
2,Max Verstappen,63
3,Sebastian Vettel,53
4,Alain Prost,51
5,Ayrton Senna,41
6,Fernando Alonso,32
7,Nigel Mansell,31
8,Jackie Stewart,27
9,Jim Clark,25


**Carreras con dos ganadores**: la vista de ganadores tiene más filas que carreras.

In [14]:
winners = data.read_table("v_race_winners")
len(winners), winners["race_id"].nunique()

(1128, 1125)

In [15]:
shared = winners[winners.duplicated("race_id", keep=False)]
shared[["year", "race_name", "driver_name", "constructor_name"]]

,year,race_name,driver_name,constructor_name
759,1957,British Grand Prix,Tony Brooks,Vanwall
763,1956,Argentine Grand Prix,Luigi Musso,Ferrari
807,1951,French Grand Prix,Luigi Fagioli,Alfa Romeo
819,1951,French Grand Prix,Juan Fangio,Alfa Romeo
820,1956,Argentine Grand Prix,Juan Fangio,Ferrari
821,1957,British Grand Prix,Stirling Moss,Vanwall


En los años 50 un piloto podía tomar el coche de un compañero durante la carrera, y los puntos se repartían entre ambos.
En estas tres carreras los dos pilotos que compartieron el coche ganador figuran como vencedores. Es un dato histórico
correcto, no un error de la carga; lo tendremos en cuenta al contar victorias.

## Conclusiones

- Se trabajó con la versión 24 del dataset de Kaggle, incluida en `data/raw/`: 14 archivos CSV con 701 433 filas y unos
  21 MB, que cubren las temporadas 1950 a 2024.
- Las 12 tablas relacionales (77 circuitos, 861 pilotos, 212 escuderías, 1125 carreras y 26 759 resultados, entre otras)
  se cargaron en una base de datos SQLite de 5.4 MB, con claves primarias, claves foráneas, índices y dos vistas de
  consulta.
- `lap_times` y `pit_stops` se mantienen como CSV: `lap_times` tiene 589 081 filas, el 84 % de todo el dataset, y solo
  se usa agregada en los análisis.
- La validación confirmó que cada tabla tiene las mismas filas que su CSV y que no hay claves foráneas inválidas.
- Las vistas permiten responder preguntas como *¿quién ganó en Monza en 2019?* (Charles Leclerc, Ferrari) con una
  consulta simple, lo que servirá para la API y el asistente LLM.
- Tres carreras de los años 50 tienen dos ganadores porque dos pilotos compartieron el coche; hay que considerarlo al
  contar victorias.